# ChurnSense-AI — 04 Model Training & Comparison

**Goal:** Train and compare four classifiers on the Phase 3 feature matrices.

| Model | Type | Why include it? |
|-------|------|------------------|
| Logistic Regression | Linear baseline | Fast, interpretable coefficients |
| Decision Tree | Single tree | Simple rules for stakeholders |
| Random Forest | Bagging ensemble | Strong tabular performer (paper's best family) |
| XGBoost | Gradient boosting | Often best AUC on churn datasets |

**Training strategy:** Baseline `X_train.csv` + **`class_weight='balanced'`** (handles imbalance without SMOTE at inference).

**Evaluation (holdout test):** accuracy, precision, recall, F1, ROC-AUC, confusion matrix, feature importance.

## 1. Setup

In [ ]:
%pip install shap
%pip install scikit-learn
%pip install xgboost

In [ ]:
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid")

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
sys.path.insert(0, str(PROJECT_ROOT))

from src.evaluate import (
    get_feature_importance,
    plot_confusion_matrix,
    plot_feature_importance,
    plot_metrics_comparison,
    plot_roc_curve,
    plot_roc_curves_comparison,
    select_best_model,
)
from src.train import (
    load_feature_matrices,
    save_trained_models,
    save_training_summary,
    train_and_evaluate_all,
)

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"

print(f"Project root: {PROJECT_ROOT}")

## 2. Load Phase 3 feature matrices

Run **`03_feature_engineering.ipynb`** first if these files are missing.

In [ ]:
required = ["X_train.csv", "X_test.csv", "y_train.csv", "y_test.csv"]
missing = [f for f in required if not (PROCESSED_DIR / f).exists()]
if missing:
    raise FileNotFoundError(
        f"Missing {missing}. Run notebooks/03_feature_engineering.ipynb first."
    )

X_train, y_train, X_test, y_test, feature_names = load_feature_matrices(
    PROCESSED_DIR, use_smote=False
)

print(f"Train: {X_train.shape} | Test: {X_test.shape}")
print(f"Train churn rate: {y_train.mean():.2%} | Test churn rate: {y_test.mean():.2%}")
print(f"Features: {len(feature_names)}")

## 3. Train all models & evaluate on test set

Modular pipeline in `src/train.py` → `train_and_evaluate_all()`.

**Why `class_weight='balanced'`?** Penalizes missing churners more heavily — better for retention campaigns than raw accuracy.

In [ ]:
models, results, comparison_df = train_and_evaluate_all(
    X_train,
    y_train,
    X_test,
    y_test,
    feature_names,
    use_class_weight=True,
)

comparison_df

### Leaderboard interpretation

- **ROC-AUC** — ranking quality across thresholds (primary metric for model selection).
- **Recall** — % of actual churners caught (critical for save offers).
- **Precision** — % of churn alerts that were correct (controls campaign cost).
- **F1** — balance of precision and recall.

## 4. Model comparison visualizations

In [ ]:
metrics_to_plot = ["roc_auc", "recall", "f1", "accuracy"]
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
axes = axes.ravel()

for ax, metric in zip(axes, metrics_to_plot):
    order = comparison_df.sort_values(metric, ascending=False)["model"]
    sns.barplot(data=comparison_df, x="model", y=metric, order=order, palette="crest", ax=ax)
    ax.set_ylim(0, 1)
    ax.set_title(metric.upper().replace("_", " "))
    ax.tick_params(axis="x", rotation=20)

plt.suptitle("ChurnSense-AI — Model comparison", y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
plot_roc_curves_comparison(y_test, results)

## 5. Per-model deep dive

Confusion matrix + classification report + ROC + top features for each model.

In [ ]:
for name in comparison_df["model"]:
    result = results[name]
    print("=" * 60)
    print(name)
    print("=" * 60)
    print(result["classification_report"])

    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

    plot_confusion_matrix(result["confusion_matrix"], f"{name} — Confusion matrix", ax=axes[0])
    plot_roc_curve(y_test, result["y_proba"], f"{name} — ROC", ax=axes[1])

    importance = get_feature_importance(models[name], feature_names, name)
    plot_feature_importance(importance, f"{name} — Top features", ax=axes[2])

    plt.tight_layout()
    plt.show()

## 6. Best model selection

**Selection rule:** Highest **ROC-AUC** on the untouched test set.

**Why not accuracy alone?** With ~74% non-churn, a lazy model looks accurate while failing retention.

**Business tie-in:** Prefer models with strong **recall** if the cost of missing a churner exceeds false alarms (typical in telecom).

In [ ]:
best_model_name = select_best_model(comparison_df, metric="roc_auc")
best_row = comparison_df.loc[comparison_df["model"] == best_model_name].iloc[0]

print(f"Best model (ROC-AUC): {best_model_name}")
print(best_row.to_string())

# If recall is the business priority, show alternate pick
best_recall_name = select_best_model(comparison_df, metric="recall")
print(f"\nHighest recall model: {best_recall_name}")

### Why the winner usually wins

| Model | Strength | Weakness |
|-------|----------|----------|
| **Logistic Regression** | Interpretable, fast | Misses non-linear patterns (contract × tenure interactions) |
| **Decision Tree** | Easy to explain | Overfits; unstable |
| **Random Forest** | Robust, strong AUC | Less interpretable than single tree |
| **XGBoost** | Often best AUC/recall | Tuning-sensitive; needs XAI (Phase 5 SHAP) |

**Expected pattern on Telco data:** Random Forest or XGBoost lead; Logistic Regression trails; Decision Tree middle but noisier.

**Business implication:** Deploy the best model to score customers weekly → CRM triggers save offers for top-decile churn probability.

## 7. Best model — feature importance (retention levers)

In [ ]:
best_importance = get_feature_importance(models[best_model_name], feature_names, best_model_name, top_n=20)
display(best_importance)

fig, ax = plt.subplots(figsize=(10, 7))
plot_feature_importance(best_importance, f"{best_model_name} — Top 20 churn drivers", ax=ax)
plt.tight_layout()
plt.show()

### Actionable drivers (typical Telco patterns)

- **Contract = Month-to-month** → offer annual lock-in discount
- **Low tenure** → onboarding / first-90-day care program
- **High MonthlyCharges** → plan review + loyalty credit
- **Electronic check payment** → autopay incentive
- **No Online Security / Tech Support** → bundle add-ons

## 8. Save models & leaderboard

In [ ]:
save_trained_models(models, MODELS_DIR, best_model_name=best_model_name)
save_training_summary(comparison_df, best_model_name, PROCESSED_DIR, MODELS_DIR)

print("Saved:")
for path in sorted(MODELS_DIR.glob("*.joblib")):
    print(f"  {path.name}")
print(f"  training_summary.json")
print(f"  {PROCESSED_DIR / 'model_comparison.csv'}")

## 9. Phase 4 summary

| Output | Location |
|--------|----------|
| All trained models | `models/*.joblib` |
| Production pick | `models/best_model.joblib` |
| Leaderboard CSV | `data/processed/model_comparison.csv` |
| JSON summary | `models/training_summary.json` |
